# Assignment 07: IMDB Text Classification in Multiple Ways

This version is simplified for Colab and assumes you already have three CSV files:

- `Train.csv`
- `Valid.csv`
- `Test.csv`

The notebook:

- loads the three ready-made splits,
- compares two preprocessing variants,
- trains a simple word-level model,
- trains a `gensim` Word2Vec representation + classifier,
- trains a character-based model,
- builds the final comparison table and answers the theory questions.


In [1]:
import importlib
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

SEED = 42
FAST_DEV_RUN = False
USE_REDUCED_DATA = True

# Balanced subset sizes per class to make training lighter on a local PC.
TRAIN_SAMPLES_PER_CLASS = 2000
VALID_SAMPLES_PER_CLASS = 500
TEST_SAMPLES_PER_CLASS = 500

WORD_EPOCHS = 3 if USE_REDUCED_DATA else 5
GENSIM_EPOCHS = 5 if USE_REDUCED_DATA else 10
CHAR_EPOCHS = 3 if USE_REDUCED_DATA else 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        return importlib.import_module(import_name)

set_seed(SEED)
print("Device:", DEVICE)
print("USE_REDUCED_DATA:", USE_REDUCED_DATA)


Device: cpu
USE_REDUCED_DATA: True


## Part A. Load CSV Splits and Compare Preprocessing


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except Exception:
    print("Colab Drive mount skipped.")

TRAIN_PATH = "/content/drive/My Drive/Train.csv"
VALID_PATH = "/content/drive/My Drive/Valid.csv"
TEST_PATH = "/content/drive/My Drive/Test.csv"

def read_split_csv(default_path, local_fallback):
    if Path(default_path).exists():
        return pd.read_csv(default_path)
    return pd.read_csv(local_fallback)

train = read_split_csv(TRAIN_PATH, "Train.csv")
valid = read_split_csv(VALID_PATH, "Valid.csv")
test = read_split_csv(TEST_PATH, "Test.csv")

print("Train shape:", train.shape)
print("Valid shape:", valid.shape)
print("Test shape:", test.shape)


Mounted at /content/drive
Google Drive mounted.
Train shape: (40000, 2)
Valid shape: (5000, 2)
Test shape: (5000, 2)


In [3]:
def normalize_dataframe(df):
    df = df.copy()

    text_candidates = ["text", "review", "sentence", "content", "comment"]
    label_candidates = ["label", "sentiment", "target", "class", "polarity"]

    text_col = next((col for col in text_candidates if col in df.columns), None)
    label_col = next((col for col in label_candidates if col in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(
            f"Could not detect text/label columns. Columns found: {list(df.columns)}"
        )

    df = df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    df["text"] = df["text"].astype(str)

    if df["label"].dtype == object:
        mapped = (
            df["label"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(
                {
                    "positive": 1,
                    "pos": 1,
                    "1": 1,
                    "negative": 0,
                    "neg": 0,
                    "0": 0,
                }
            )
        )
        if mapped.isna().any():
            raise ValueError("Label mapping failed. Expected binary sentiment labels.")
        df["label"] = mapped.astype(int)
    else:
        df["label"] = df["label"].astype(int)

    df["label_name"] = df["label"].map({0: "negative", 1: "positive"})
    return df

def take_balanced_subset(df, per_class, seed=SEED):
    parts = []
    for label_value in sorted(df["label"].unique()):
        class_df = df[df["label"] == label_value]
        sample_size = min(per_class, len(class_df))
        parts.append(class_df.sample(n=sample_size, random_state=seed))
    return (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

train_df = normalize_dataframe(train)
val_df = normalize_dataframe(valid)
test_df = normalize_dataframe(test)

print("Original sizes:", len(train_df), len(val_df), len(test_df))

if USE_REDUCED_DATA:
    train_df = take_balanced_subset(train_df, TRAIN_SAMPLES_PER_CLASS)
    val_df = take_balanced_subset(val_df, VALID_SAMPLES_PER_CLASS)
    test_df = take_balanced_subset(test_df, TEST_SAMPLES_PER_CLASS)

if FAST_DEV_RUN:
    train_df = take_balanced_subset(train_df, 300)
    val_df = take_balanced_subset(val_df, 100)
    test_df = take_balanced_subset(test_df, 100)

print("Working sizes:", len(train_df), len(val_df), len(test_df))

def strip_html(text):
    text = text.replace("<br />", " ")
    return re.sub(r"<[^>]+>", " ", text)

def normalize_space(text):
    return re.sub(r"\s+", " ", text).strip()

def word_count(text):
    clean = normalize_space(strip_html(text))
    return len(re.findall(r"\b\w+\b", clean))

for df in [train_df, val_df, test_df]:
    df["word_count"] = df["text"].map(word_count)

split_table = pd.DataFrame(
    [
        {"split": "train", "size": len(train_df), "positive_pct": round(train_df["label"].mean() * 100, 2), "avg_review_len": round(train_df["word_count"].mean(), 2)},
        {"split": "validation", "size": len(val_df), "positive_pct": round(val_df["label"].mean() * 100, 2), "avg_review_len": round(val_df["word_count"].mean(), 2)},
        {"split": "test", "size": len(test_df), "positive_pct": round(test_df["label"].mean() * 100, 2), "avg_review_len": round(test_df["word_count"].mean(), 2)},
    ]
)
display(split_table)


Original sizes: 40000 5000 5000
Working sizes: 4000 1000 1000


,split,size,positive_pct,avg_review_len
0,train,4000,50.0,237.67
1,validation,1000,50.0,231.90
2,test,1000,50.0,238.82


In [4]:
sample_examples = pd.concat(
    [
        train_df[train_df["label"] == 1].head(2),
        train_df[train_df["label"] == 0].head(2),
    ],
    ignore_index=True,
).copy()
sample_examples["snippet"] = sample_examples["text"].map(
    lambda x: normalize_space(strip_html(x))[:350] + "..."
)
display(sample_examples[["label_name", "snippet"]])


,label_name,snippet
0,positive,"Story-wise this isn't among the best or most cleverly written Columbo movie but the movie is extremely well made, with excellent directi..."
1,positive,"Yes, it is a bit cheesy. But it's suspenseful and entertaining, and one of my favorites; there are some excellent actors in the film, an..."
2,negative,"This movie is standard goofy sci-fi fare from the 50s. In its favor, the plot does manage to pull off an alien invasion without actually..."
3,negative,"Made and released at the time when the internet was just becoming huge, this is a storyline Hitchcock would have loved. Sadly, Hitchcock..."


In [5]:
NEGATION_WORDS = {"no", "not", "nor", "never", "n't"}
STOPWORDS_V2 = set(ENGLISH_STOP_WORDS) - NEGATION_WORDS

def preprocess_v1(text):
    text = normalize_space(strip_html(text).lower())
    text = re.sub(r"[^a-z0-9']+", " ", text)
    return [token for token in text.split() if token]

def preprocess_v2(text):
    text = normalize_space(strip_html(text).lower())
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?|\d+|[!?]+", text)
    return [token for token in tokens if token not in STOPWORDS_V2]

preprocessors = {
    "variant_1_basic": preprocess_v1,
    "variant_2_regex_stopwords": preprocess_v2,
}

tokenized = {}
for name, fn in preprocessors.items():
    tokenized[name] = {
        "train": train_df["text"].map(fn).tolist(),
        "validation": val_df["text"].map(fn).tolist(),
        "test": test_df["text"].map(fn).tolist(),
    }

def summarize_tokens(token_lists):
    counter = Counter(token for row in token_lists for token in row)
    return counter

preprocessing_rows = []
for name, splits in tokenized.items():
    counter = summarize_tokens(splits["train"])
    preprocessing_rows.append(
        {
            "preprocessing": name,
            "vocab_size_before_cutoff": len(counter),
            "10_most_frequent": counter.most_common(10),
            "10_rare_tokens": sorted([token for token, freq in counter.items() if freq == 1])[:10],
        }
    )

    display(Markdown(f"### {name}"))
    display(pd.DataFrame({"tokenized_example": [tokens[:25] for tokens in splits["train"][:5]]}))
    print("Vocab size before cutoff:", len(counter))
    print("10 most frequent tokens:", counter.most_common(10))
    print("10 rare tokens:", sorted([token for token, freq in counter.items() if freq == 1])[:10])
    print("-" * 100)

preprocessing_summary_df = pd.DataFrame(preprocessing_rows)
display(preprocessing_summary_df[["preprocessing", "vocab_size_before_cutoff"]])


### variant_1_basic

,tokenized_example
0,"[this, movie, is, standard, goofy, sci, fi, fare, from, the, 50s, in, its, favor, the, plot, does, manage, to, pull, off, an, alien, inv..."
1,"[story, wise, this, isn't, among, the, best, or, most, cleverly, written, columbo, movie, but, the, movie, is, extremely, well, made, wi..."
2,"[made, and, released, at, the, time, when, the, internet, was, just, becoming, huge, this, is, a, storyline, hitchcock, would, have, lov..."
3,"[yes, it, is, a, bit, cheesy, but, it's, suspenseful, and, entertaining, and, one, of, my, favorites, there, are, some, excellent, actor..."
4,"[convoluted, infuriating, and, implausible, fay, grim, is, hard, to, sit, through, but, parker, posey, is, really, the, only, actress, w..."


Vocab size before cutoff: 38723
10 most frequent tokens: [('the', 53688), ('and', 26469), ('a', 26251), ('of', 23381), ('to', 21703), ('is', 17234), ('in', 14963), ('it', 12616), ('i', 12388), ('this', 12070)]
10 rare tokens: ["''anchors", "''from", "''gaslight''", "''inuyasha''", "''maison", "''swingin'", "''the", "''till", "'01", "'04"]
----------------------------------------------------------------------------------------------------


### variant_2_regex_stopwords

,tokenized_example
0,"[movie, standard, goofy, sci, fi, fare, 50, s, favor, plot, does, manage, pull, alien, invasion, actually, producing, aliens, come, alie..."
1,"[story, wise, isn't, best, cleverly, written, columbo, movie, movie, extremely, excellent, directing, truly, fine, acting, especially, a..."
2,"[released, time, internet, just, huge, storyline, hitchcock, loved, sadly, hitchcock, wasn't, make, we're, left, occasionally, suspensef..."
3,"[yes, bit, cheesy, it's, suspenseful, entertaining, favorites, excellent, actors, film, commendable, job, given, limitations, plot, char..."
4,"[convoluted, infuriating, implausible, fay, grim, hard, sit, parker, posey, really, actress, story, run, she's, touching, funny, cunning..."


Vocab size before cutoff: 36882
10 most frequent tokens: [('movie', 6982), ('film', 6378), ('not', 4914), ('like', 3232), ('just', 2822), ("it's", 2594), ('!', 2573), ('?', 2380), ('good', 2376), ('no', 2070)]
10 rare tokens: ['!!!!!!!!', '!!!!!!!!!!', '!!!!!!!!!!!!!!!!!!!!!!!', '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!', '!!!???', '!!??!!!', '!?!?!???', '0077247', '03', '04']
----------------------------------------------------------------------------------------------------


,preprocessing,vocab_size_before_cutoff
0,variant_1_basic,38723
1,variant_2_regex_stopwords,36882


Lowercasing should help because sentiment words usually keep the same meaning regardless of case, and it reduces vocabulary fragmentation. Removing simple punctuation may help the model focus on content words, but it can also remove emphasis such as repeated exclamation marks. Stopword removal may reduce noise, yet sentiment tasks are sensitive to negation, so deleting short function words can hurt. Because of that, the more aggressive pipeline may become cleaner but also risk losing polarity cues like `not good`.


In [6]:
def join_tokens(token_lists):
    return [" ".join(tokens) if tokens else "<EMPTY>" for tokens in token_lists]

baseline_rows = []
for name, splits in tokenized.items():
    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(join_tokens(splits["train"]))
    X_val = vectorizer.transform(join_tokens(splits["validation"]))
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(X_train, train_df["label"])
    val_pred = clf.predict(X_val)
    baseline_rows.append(
        {
            "Preprocessing": name,
            "Val Acc": round(accuracy_score(val_df["label"], val_pred), 4),
            "Vectorizer Vocab": len(vectorizer.vocabulary_),
        }
    )

preprocessing_baseline_df = pd.DataFrame(baseline_rows).sort_values("Val Acc", ascending=False).reset_index(drop=True)
SELECTED_PREPROCESSING = preprocessing_baseline_df.iloc[0]["Preprocessing"]
display(preprocessing_baseline_df)
print("Selected preprocessing:", SELECTED_PREPROCESSING)


,Preprocessing,Val Acc,Vectorizer Vocab
0,variant_1_basic,0.859,20000
1,variant_2_regex_stopwords,0.855,20000


Selected preprocessing: variant_1_basic


## Part B. Word-Level Integer Encoding Model


In [7]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MIN_FREQ = 3

train_tokens = tokenized[SELECTED_PREPROCESSING]["train"]
val_tokens = tokenized[SELECTED_PREPROCESSING]["validation"]
test_tokens = tokenized[SELECTED_PREPROCESSING]["test"]

counter = Counter(token for row in train_tokens for token in row)
vocab_tokens = [token for token, freq in counter.items() if freq >= MIN_FREQ]
vocab_tokens = sorted(vocab_tokens)
idx_to_token = [PAD_TOKEN, UNK_TOKEN] + vocab_tokens
token_to_idx = {token: idx for idx, token in enumerate(idx_to_token)}
PAD_IDX = token_to_idx[PAD_TOKEN]
UNK_IDX = token_to_idx[UNK_TOKEN]

MAX_LEN = int(np.percentile([len(row) for row in train_tokens], 95))
MAX_LEN = max(50, min(MAX_LEN, 400))

print("min_freq =", MIN_FREQ)
print("maximum sequence length =", MAX_LEN)
print("vocabulary size =", len(idx_to_token))
print("padding and truncation rule = truncate to MAX_LEN, then right-pad with <PAD>")

def encode_tokens(tokens, max_len):
    ids = [token_to_idx.get(token, UNK_IDX) for token in tokens[:max_len]]
    if not ids:
        ids = [UNK_IDX]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids

class TextDataset(Dataset):
    def __init__(self, token_lists, labels, max_len):
        self.token_lists = token_lists
        self.labels = labels.tolist() if hasattr(labels, "tolist") else list(labels)
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(encode_tokens(self.token_lists[idx], self.max_len), dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long),
        }

class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        mask = (x != PAD_IDX).unsqueeze(-1)
        summed = (emb * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean_emb = summed / counts
        return self.fc(mean_emb)

def make_loader(dataset, batch_size=128, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def evaluate_model(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            logits = model(x)
            pred = logits.argmax(dim=1)
            pred_prob = torch.softmax(logits, dim=1).max(dim=1).values
            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())
            probs.extend(pred_prob.cpu().numpy())
    return {
        "acc": accuracy_score(labels, preds),
        "preds": np.array(preds),
        "labels": np.array(labels),
        "confidences": np.array(probs),
    }

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_state = None
    best_val_acc = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_metrics = evaluate_model(model, train_loader)
        val_metrics = evaluate_model(model, val_loader)
        history.append(
            {
                "epoch": epoch,
                "train_loss": round(total_loss / len(train_loader), 4),
                "train_acc": round(train_metrics["acc"], 4),
                "val_acc": round(val_metrics["acc"], 4),
            }
        )
        print(history[-1])

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return pd.DataFrame(history)

word_train_ds = TextDataset(train_tokens, train_df["label"], MAX_LEN)
word_val_ds = TextDataset(val_tokens, val_df["label"], MAX_LEN)
word_test_ds = TextDataset(test_tokens, test_df["label"], MAX_LEN)

word_train_loader = make_loader(word_train_ds, shuffle=True)
word_val_loader = make_loader(word_val_ds)
word_test_loader = make_loader(word_test_ds)

word_model = MeanEmbeddingClassifier(len(idx_to_token), 100, 2, PAD_IDX).to(DEVICE)
word_history_df = train_model(word_model, word_train_loader, word_val_loader, epochs=2 if FAST_DEV_RUN else WORD_EPOCHS)
display(word_history_df)

word_train_metrics = evaluate_model(word_model, word_train_loader)
word_val_metrics = evaluate_model(word_model, word_val_loader)
word_test_metrics = evaluate_model(word_model, word_test_loader)

display(
    pd.DataFrame(
        [
            {"split": "train", "accuracy": round(word_train_metrics["acc"], 4)},
            {"split": "validation", "accuracy": round(word_val_metrics["acc"], 4)},
            {"split": "test", "accuracy": round(word_test_metrics["acc"], 4)},
        ]
    )
)


min_freq = 3
maximum sequence length = 400
vocabulary size = 16043
padding and truncation rule = truncate to MAX_LEN, then right-pad with <PAD>
{'epoch': 1, 'train_loss': 0.6919, 'train_acc': 0.6, 'val_acc': 0.549}
{'epoch': 2, 'train_loss': 0.6776, 'train_acc': 0.6765, 'val_acc': 0.624}
{'epoch': 3, 'train_loss': 0.661, 'train_acc': 0.7057, 'val_acc': 0.657}


,epoch,train_loss,train_acc,val_acc
0,1,0.6919,0.6000,0.549
1,2,0.6776,0.6765,0.624
2,3,0.6610,0.7057,0.657


,split,accuracy
0,train,0.7057
1,validation,0.6570
2,test,0.6580


In [8]:
label_to_name = {0: "negative", 1: "positive"}

word_examples = test_df.copy()
word_examples["pred"] = word_test_metrics["preds"]
word_examples["pred_name"] = word_examples["pred"].map(label_to_name)
word_examples["confidence"] = np.round(word_test_metrics["confidences"], 4)
word_examples["correct"] = word_examples["pred"] == word_examples["label"]
word_examples["snippet"] = word_examples["text"].map(lambda x: normalize_space(strip_html(x))[:320] + "...")

display(Markdown("### 3 correct predictions"))
display(word_examples[word_examples["correct"]].head(3)[["label_name", "pred_name", "confidence", "snippet"]])

display(Markdown("### 3 incorrect predictions"))
display(word_examples[~word_examples["correct"]].head(3)[["label_name", "pred_name", "confidence", "snippet"]])


### 3 correct predictions

,label_name,pred_name,confidence,snippet
0,positive,positive,0.6186,"This is a great movie. The best role Peter Strauss ever did. The music is good, the message harsh, the actors great and the story is bot..."
1,positive,positive,0.6093,"Loved this movie, what a hoot. Rupert and Julie are great together with Rupert being almost poker faced against Julie's animation, which..."
2,positive,positive,0.5582,One of the best true-crime movies ever made and very faithful to Truman Capote's book which invented the true-crime novel genre. Hauntin...


### 3 incorrect predictions

,label_name,pred_name,confidence,snippet
4,negative,positive,0.5374,"Critics are falling over themselves within the Weinstein's Sphere of Influence to praise this ugly, misguided and repellent adaptation o..."
11,negative,positive,0.5191,"This film may have been the biggest let-down I've experienced in renting movies based on IMDb reviews. Overall, I simply found this to b..."
15,positive,negative,0.5088,"Preston Waters is off to a bad summer. Besides his birthday coming up, nothing else looks promising. First he has to share his own room ..."


## Part C. Gensim Word Embedding Model


In [9]:
ensure_package("gensim")
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=3,
    workers=1,
    sg=1,
    epochs=2 if FAST_DEV_RUN else GENSIM_EPOCHS,
    seed=SEED,
)

known_mean_vector = w2v_model.wv.vectors.mean(axis=0)
random_vector = np.random.default_rng(SEED).normal(0, 0.1, size=100).astype(np.float32)

def review_to_vector(tokens, strategy):
    vectors = []
    found = 0
    missing = 0

    for token in tokens:
        if token in w2v_model.wv:
            vectors.append(w2v_model.wv[token])
            found += 1
        else:
            missing += 1
            if strategy == "skip_missing":
                continue
            if strategy == "avg_known_vector":
                vectors.append(known_mean_vector)
            elif strategy == "reused_random_vector":
                vectors.append(random_vector)

    if not vectors:
        vectors = [known_mean_vector]

    return np.mean(np.vstack(vectors), axis=0), found, missing

def build_dense_features(token_lists, strategy):
    rows = []
    total_found = 0
    total_missing = 0
    for tokens in token_lists:
        vec, found, missing = review_to_vector(tokens, strategy)
        rows.append(vec)
        total_found += found
        total_missing += missing
    return np.vstack(rows), total_found, total_missing

gensim_rows = []
for strategy in ["skip_missing", "avg_known_vector", "reused_random_vector"]:
    X_train, train_found, train_missing = build_dense_features(train_tokens, strategy)
    X_val, val_found, val_missing = build_dense_features(val_tokens, strategy)
    X_test, test_found, test_missing = build_dense_features(test_tokens, strategy)

    clf = LogisticRegression(max_iter=1200, random_state=SEED)
    clf.fit(X_train, train_df["label"])

    train_acc = accuracy_score(train_df["label"], clf.predict(X_train))
    val_acc = accuracy_score(val_df["label"], clf.predict(X_val))
    test_acc = accuracy_score(test_df["label"], clf.predict(X_test))

    gensim_rows.append(
        {
            "OOV Strategy": strategy,
            "Found Tokens (val)": val_found,
            "Missing Tokens (val)": val_missing,
            "Val Acc": round(val_acc, 4),
            "Test Acc": round(test_acc, 4),
            "Train Acc": round(train_acc, 4),
        }
    )

gensim_results_df = pd.DataFrame(gensim_rows).sort_values("Val Acc", ascending=False).reset_index(drop=True)
BEST_GENSIM_ROW = gensim_results_df.iloc[0]
display(gensim_results_df)


,OOV Strategy,Found Tokens (val),Missing Tokens (val),Val Acc,Test Acc,Train Acc
0,skip_missing,217370,10004,0.800,0.791,0.7947
1,avg_known_vector,217370,10004,0.800,0.792,0.7937
2,reused_random_vector,217370,10004,0.797,0.796,0.7945


## Part D. Character-Based Classification


In [10]:
CHAR_PAD = "<PAD>"
CHAR_UNK = "<UNK>"

train_char_texts = train_df["text"].map(lambda x: normalize_space(strip_html(x).lower())).tolist()
val_char_texts = val_df["text"].map(lambda x: normalize_space(strip_html(x).lower())).tolist()
test_char_texts = test_df["text"].map(lambda x: normalize_space(strip_html(x).lower())).tolist()

char_counter = Counter(ch for text in train_char_texts for ch in text)
char_vocab = [ch for ch, freq in char_counter.items() if freq >= 2]
char_vocab = sorted(char_vocab)
idx_to_char = [CHAR_PAD, CHAR_UNK] + char_vocab
char_to_idx = {ch: idx for idx, ch in enumerate(idx_to_char)}
CHAR_PAD_IDX = char_to_idx[CHAR_PAD]
CHAR_UNK_IDX = char_to_idx[CHAR_UNK]

CHAR_MAX_LEN = int(np.percentile([len(text) for text in train_char_texts], 95))
CHAR_MAX_LEN = max(300, min(CHAR_MAX_LEN, 1200))

print("maximum character length =", CHAR_MAX_LEN)
print("character vocabulary size =", len(idx_to_char))

def encode_chars(text, max_len):
    ids = [char_to_idx.get(ch, CHAR_UNK_IDX) for ch in text[:max_len]]
    if not ids:
        ids = [CHAR_UNK_IDX]
    if len(ids) < max_len:
        ids += [CHAR_PAD_IDX] * (max_len - len(ids))
    return ids

class CharDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.texts = texts
        self.labels = labels.tolist() if hasattr(labels, "tolist") else list(labels)
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(encode_chars(self.texts[idx], self.max_len), dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long),
        }

class SimpleCharCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv = nn.Conv1d(embed_dim, 128, kernel_size=5, padding=2)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_out = torch.relu(self.conv(emb))
        pooled = torch.amax(conv_out, dim=2)
        return self.fc(pooled)

char_train_ds = CharDataset(train_char_texts, train_df["label"], CHAR_MAX_LEN)
char_val_ds = CharDataset(val_char_texts, val_df["label"], CHAR_MAX_LEN)
char_test_ds = CharDataset(test_char_texts, test_df["label"], CHAR_MAX_LEN)

char_train_loader = make_loader(char_train_ds, shuffle=True)
char_val_loader = make_loader(char_val_ds)
char_test_loader = make_loader(char_test_ds)

char_model = SimpleCharCNN(len(idx_to_char), 32, 2, CHAR_PAD_IDX).to(DEVICE)
char_history_df = train_model(char_model, char_train_loader, char_val_loader, epochs=2 if FAST_DEV_RUN else CHAR_EPOCHS)
display(char_history_df)

char_train_metrics = evaluate_model(char_model, char_train_loader)
char_val_metrics = evaluate_model(char_model, char_val_loader)
char_test_metrics = evaluate_model(char_model, char_test_loader)

display(
    pd.DataFrame(
        [
            {"split": "train", "accuracy": round(char_train_metrics["acc"], 4)},
            {"split": "validation", "accuracy": round(char_val_metrics["acc"], 4)},
            {"split": "test", "accuracy": round(char_test_metrics["acc"], 4)},
        ]
    )
)


maximum character length = 1200
character vocabulary size = 103
{'epoch': 1, 'train_loss': 0.675, 'train_acc': 0.6172, 'val_acc': 0.605}
{'epoch': 2, 'train_loss': 0.6062, 'train_acc': 0.7923, 'val_acc': 0.747}
{'epoch': 3, 'train_loss': 0.5523, 'train_acc': 0.808, 'val_acc': 0.757}


,epoch,train_loss,train_acc,val_acc
0,1,0.6750,0.6172,0.605
1,2,0.6062,0.7923,0.747
2,3,0.5523,0.8080,0.757


,split,accuracy
0,train,0.808
1,validation,0.757
2,test,0.753


In [11]:
comparison_examples = test_df.copy()
comparison_examples["word_pred"] = word_test_metrics["preds"]
comparison_examples["char_pred"] = char_test_metrics["preds"]
comparison_examples["word_ok"] = comparison_examples["word_pred"] == comparison_examples["label"]
comparison_examples["char_ok"] = comparison_examples["char_pred"] == comparison_examples["label"]
comparison_examples["word_pred_name"] = comparison_examples["word_pred"].map(label_to_name)
comparison_examples["char_pred_name"] = comparison_examples["char_pred"].map(label_to_name)
comparison_examples["snippet"] = comparison_examples["text"].map(lambda x: normalize_space(strip_html(x))[:320] + "...")

diff_examples = pd.concat(
    [
        comparison_examples[(comparison_examples["char_ok"]) & (~comparison_examples["word_ok"])].head(2),
        comparison_examples[(comparison_examples["word_ok"]) & (~comparison_examples["char_ok"])].head(2),
    ],
    ignore_index=True,
).drop_duplicates(subset=["snippet"]).head(3)

display(diff_examples[["label_name", "word_pred_name", "char_pred_name", "word_ok", "char_ok", "snippet"]])


,label_name,word_pred_name,char_pred_name,word_ok,char_ok,snippet
0,negative,positive,negative,False,True,"This film may have been the biggest let-down I've experienced in renting movies based on IMDb reviews. Overall, I simply found this to b..."
1,negative,positive,negative,False,True,"*WARNING. THERE MIGHT BE SPOILERS AHEAD, IF YOU CARE.* Okay, the basic premise of this homegrown Texas film is: College kids + spookhous..."
2,positive,positive,negative,True,False,"""The Deer Hunter's"" success with critics and publics alike led United Artists to give Cimino carte-blanche on ""Heaven's Gate,"" an epic W..."


## Part E. Compare All Approaches


In [12]:
comparison_df = pd.DataFrame(
    [
        {
            "Model / Representation": "Word IDs + learned embedding",
            "Preprocessing": SELECTED_PREPROCESSING,
            "OOV Strategy": "<UNK>",
            "Vocab Size": len(idx_to_token),
            "Max Length": MAX_LEN,
            "Train Acc": round(word_train_metrics["acc"], 4),
            "Val Acc": round(word_val_metrics["acc"], 4),
            "Test Acc": round(word_test_metrics["acc"], 4),
            "Notes": "Embedding + mean pooling + linear layer",
        },
        {
            "Model / Representation": "Gensim embeddings",
            "Preprocessing": SELECTED_PREPROCESSING,
            "OOV Strategy": BEST_GENSIM_ROW["OOV Strategy"],
            "Vocab Size": len(w2v_model.wv),
            "Max Length": "all tokens",
            "Train Acc": BEST_GENSIM_ROW["Train Acc"],
            "Val Acc": BEST_GENSIM_ROW["Val Acc"],
            "Test Acc": BEST_GENSIM_ROW["Test Acc"],
            "Notes": "Word2Vec + mean pooling + LogisticRegression",
        },
        {
            "Model / Representation": "Character-based model",
            "Preprocessing": "lowercase + html cleanup",
            "OOV Strategy": "<UNK>",
            "Vocab Size": len(idx_to_char),
            "Max Length": CHAR_MAX_LEN,
            "Train Acc": round(char_train_metrics["acc"], 4),
            "Val Acc": round(char_val_metrics["acc"], 4),
            "Test Acc": round(char_test_metrics["acc"], 4),
            "Notes": "Character CNN",
        },
    ]
).sort_values("Val Acc", ascending=False).reset_index(drop=True)

display(comparison_df)

print("Which representation worked best?", comparison_df.iloc[0]["Model / Representation"])
print("Which preprocessing choice mattered most?", preprocessing_baseline_df.iloc[0]["Preprocessing"])
print("Which unknown-word strategy worked best?", BEST_GENSIM_ROW["OOV Strategy"])
print("Which model is best for misspellings or rare words? Character-based model.")


,Model / Representation,Preprocessing,OOV Strategy,Vocab Size,Max Length,Train Acc,Val Acc,Test Acc,Notes
0,Gensim embeddings,variant_1_basic,skip_missing,16041,all tokens,0.7947,0.800,0.791,Word2Vec + mean pooling + LogisticRegression
1,Character-based model,lowercase + html cleanup,<UNK>,103,1200,0.8080,0.757,0.753,Character CNN
2,Word IDs + learned embedding,variant_1_basic,<UNK>,16043,400,0.7057,0.657,0.658,Embedding + mean pooling + linear layer


Which representation worked best? Gensim embeddings
Which preprocessing choice mattered most? variant_1_basic
Which unknown-word strategy worked best? skip_missing
Which model is best for misspellings or rare words? Character-based model.


## Follow-Up Answers

1. **What is the difference between a token, a vocabulary item, and an embedding vector?**  
A token is one text unit after preprocessing. A vocabulary item is a unique token stored in the model dictionary. An embedding vector is the dense numeric representation assigned to that vocabulary item.

2. **Why should the vocabulary be built only from the training data?**  
Because using validation or test text would leak information from data that should stay unseen during training.

3. **What does `<UNK>` represent?**  
It is the fallback symbol for unknown or filtered-out tokens.

4. **Why can skipping unknown words be dangerous for sentiment classification?**  
Because a rare unknown word can still carry strong sentiment, and skipping it can remove the main polarity cue.

5. **Why might a character-based model handle unseen words better than a word-level model?**  
It can learn from spelling patterns inside the word, even if the full word never appeared in training.

6. **What information can be lost when representing a review by the mean of its word vectors?**  
Word order, emphasis, and negation structure can disappear.

7. **Why can stopword removal sometimes hurt sentiment classification?**  
Because negation and short functional words are often critical for polarity.

8. **Which approach was most sensitive to preprocessing in your experiments?**  
Use the validation scores above. Usually the word-based approaches are more sensitive than the character-based one.
